## **Análisis de Interpretabilidad de los Modelos Seleccionados**

El objetivo de este notebook es analizar el peso que asigna cada modelo a cada modalidad (vídeo, audio y texto) a la hora de detectar el estrés. 

La estructura de este notebook es la siguiente:

1. **Interpretabilidad del Modelo Global (Fusión mediante Atención)**: Para la partición de test del dataset global, obtenemos los **pesos atencionales** que asigna el modelo (correspondientes a la capa de atención en la arquitectura) a cada modalidad. Este análisis de interpretabilidad lo hacemos directamente en `evaluate.py` ya que necesitamos iterar por las muestras del dataset y recoger los pesos de atención muestra a muestra (devolviendo el promedio final por cada modalidad). 

2. **Interpretabilidad de MELD (Fusión Tardía mediante Regresión Logística)**:
Esta arquitectura cuenta con una capa final adicional (diferente al resto de técnicas de fusión tardía) con 3 neuronas a la entrada 1 a la salida. Para el análisis de interpretabilidad, extraemos los pesos aprendidos (que son tres escalares) de la capa de `self.logistic_fusion`, que representan la importancia relativa de cada modalidad aprendida por la red durante el entrenamiento. 

3. **Interpretabilidad de IEMOCAP (Fusión Temprana)**:
Al concatenarse directamente las características extraídas de cada modalidad, lo que haremos será interceptar los pesos de la **primera capa lineal** que recibe ese primer vector concatenado, y sumaremos el valor absoluto de los pesos conectados a cada sección (modalidad). Si las conexiones que salen, por ejemplo, de la sección de "Audio" son más pesadas que las del "Vídeo", significa que la red aprendió durante el entrenamiento a darle más importancia a la voz de los actores que a su cara, por ejemplo.

**NOTA**: En el caso del análisis de interpretabilidad de la regresión logística y la fusión temprana, al no necesitar el dataset ya que los pesos son estáticos y no dinámicos como en el caso de la atención, lo calculamos en este notebook directamente. 


In [1]:
# Importamos las librerías necesarias:

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import subprocess
from models.fusion_strategies import EarlyFusion, LateFusion
import pandas as pd


# Configuración de GPU/CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

Dispositivo: cuda


---
### **Interpretabilidad del Modelo Global (Fusión mediante Atención)**

In [ ]:
path_pesos = "pesos/inter_multimodal_ajuste/attention/global/pesos_modelo_estres_global_attention_resnet32_wav2vec7s_roberta32_p512_h128_lr5e-05_do0.3.pth"

comando = (f"python evaluate.py "
           f"--model_path {path_pesos} "
           f"--eval_dataset global "
           f"--split test "
           f"--fusion attention "
           f"--video resnet "
           f"--video_frames 32 "
           f"--audio wav2vec "
           f"--audio_len 7 "
           f"--text roberta32 "
           f"--proj_dim 512 "
           f"--hidden_mlp 128 "
           f"--dropout 0.3 "
           f"--interp_atencion True ") ##
           
subprocess.run(comando, shell=True)

100%|██████████| 4257/4257 [00:19<00:00, 216.82it/s]


MÉTRICAS DEL MODELO EN TEST:
Parámetros Totales: 8,928,514
Parámetros Entrenables: 8,928,514
Tiempo medio de Inferencia: 3.47 ms / muestra
ROC-AUC Score: 0.7837
F1 Macro: 0.6891
F1 Weighted: 0.7690
Accuracy (en %): 75.69%

RESUMEN DE INTERPRETABILIDAD (Atención)
 Vídeo : 29.53%
 Audio : 18.15%
 Texto : 52.32%

Vista previa de las 5 primeras muestras (CSV):
 Peso_Video  Peso_Audio  Peso_Texto  Label_Real  Prediccion
   0.420953    0.192588    0.386460         0.0         0.0
   0.277436    0.150279    0.572285         1.0         0.0
   0.205603    0.208796    0.585600         0.0         0.0
   0.415211    0.184675    0.400114         0.0         0.0
   0.300352    0.188185    0.511462         0.0         0.0


CompletedProcess(args='python evaluate_multimodal.py --model_path pesos/pesos_entrenamiento_ajuste_attention/global/pesos_modelo_estres_global_attention_resnet32_wav2vec7s_roberta32_p512_h128_lr5e-05_do0.3.pth --eval_dataset global --split test --fusion attention --video resnet --video_frames 32 --audio wav2vec --audio_len 7 --text roberta32 --proj_dim 512 --hidden_mlp 128 --dropout 0.3 --interp_atencion True ', returncode=0)

----
### **Interpretabilidad de MELD (Fusión Tardía mediante Regresión Logística)**

In [4]:
# Cargamos el modelo ganador de MELD (Late Fusion Logística)
modelo_meld = LateFusion(
    visual_dim=2048,
    audio_dim=768,
    text_dim=768,
    proj_dim=512,
    hidden_mlp=128,
    dropout_prob=0.5,
    fusion_mode='logistica'
).to(device)

ruta_pesos_meld = "pesos/inter_multimodal_ajuste/late/logistica/MELD/pesos_modelo_estres_MELD_late_logistica_resnet32_wav2vec7s_roberta32_p512_h128_lr0.0001_do0.5.pth"
modelo_meld.load_state_dict(torch.load(ruta_pesos_meld, map_location=device))
modelo_meld.eval()

# Extraemos los pesos aprendidos de la capa de regresión logística:
pesos = modelo_meld.logistic_fusion.weight.data.cpu().numpy().flatten()
pesos_abs = np.abs(pesos)
importancia = pesos_abs / pesos_abs.sum() * 100

print("\nANÁLISIS DE INTERPRETABILIDAD MELD (LATE FUSION - REGRESIÓN LOGÍSTICA)")
print("Importancia relativa de cada modalidad (pesos aprendidos):")
print(f"Vídeo : {importancia[0]:.2f}% (peso bruto: {pesos[0]:.4f})")
print(f"Audio : {importancia[1]:.2f}% (peso bruto: {pesos[1]:.4f})")
print(f"Texto : {importancia[2]:.2f}% (peso bruto: {pesos[2]:.4f})")


ANÁLISIS DE INTERPRETABILIDAD MELD (LATE FUSION - REGRESIÓN LOGÍSTICA)
Importancia relativa de cada modalidad (pesos aprendidos):
Vídeo : 7.49% (peso bruto: -0.0771)
Audio : 34.90% (peso bruto: -0.3590)
Texto : 57.60% (peso bruto: 0.5925)


---
### **Interpretabilidad de IEMOCAP (Fusión Temprana)**

En este caso, directamente lo que hacemos es cargar el modelo (con los mismos hiperparámetros), extraemos la matriz de pesos de la primera capa, sumamos el valor absoluto de todas las conexiones por cada modalidad y vemos a qué "trozo" del vector le ha dado más peso la red. La obtención de los pesos o importancia que asigna el modelo a cada modalidad para la fusión temprana viene resumido en la siguiente imagen:

<img src="../figuras/Fig_Analisis_Interpretabilidad_Fusion_Temprana.png" width="1000">

In [5]:
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

# INSTANCIAMOS EL MODELO CAMPEÓN DE IEMOCAP
# Usamos los hiperparámetros ganadores (proj=512, hidden=128)
modelo_early = EarlyFusion(
    visual_dim=2048,
    audio_dim=768,
    text_dim=768,
    proj_dim=512, 
    hidden_mlp=128, 
    dropout_prob=0.3).to(device)

ruta_pesos_iemocap = "pesos/inter_multimodal_ajuste/early/IEMOCAP/pesos_modelo_estres_IEMOCAP_early_resnet32_wav2vec7s_roberta32_p512_h128_lr0.0001_do0.3.pth"
modelo_early.load_state_dict(torch.load(ruta_pesos_iemocap, map_location=device))
modelo_early.eval()

#  EXTRACCIÓN DE LA MATRIZ DE PESOS
# En EarlyFusion, el clasificador MLP recibe en su primera capa proj_dim * 3 entradas
# Extraemos los pesos de la primera capa lineal: self.mlp_classifier[0]
# Shape esperado: [512, 1536] (512 neuronas de salida, 1536 de entrada)
matriz_pesos = modelo_early.mlp[0].weight.data.cpu().numpy()

# CÁLCULO DE IMPORTANCIA 
# Sumamos el valor absoluto de los pesos que salen de cada una de las 1536 neuronas de entrada
importancia_neuronas = np.sum(np.abs(matriz_pesos), axis=0) # Shape: [1536]

# Dividimos el vector gigante en los 3 trozos originales (proj_dim = 512)
proj_dim = 512
imp_video = np.sum(importancia_neuronas[0 : proj_dim])
imp_audio = np.sum(importancia_neuronas[proj_dim : proj_dim * 2])
imp_texto = np.sum(importancia_neuronas[proj_dim * 2 : proj_dim * 3])

# Convertimos a porcentajes para poder interpretarlo más fácilmente:
importancia_total = imp_video + imp_audio + imp_texto
porc_video = (imp_video / importancia_total) * 100
porc_audio = (imp_audio / importancia_total) * 100
porc_texto = (imp_texto / importancia_total) * 100

print("\nANÁLISIS DE INTERPRETABILIDAD IEMOCAP (EARLY FUSION)")
print("Peso asignado por la red durante el entrenamiento a cada dominio:")
print(f"Vídeo : {porc_video:.2f}%")
print(f"Audio : {porc_audio:.2f}%")
print(f"Texto : {porc_texto:.2f}%")


ANÁLISIS DE INTERPRETABILIDAD IEMOCAP (EARLY FUSION)
Peso asignado por la red durante el entrenamiento a cada dominio:
Vídeo : 33.05%
Audio : 32.75%
Texto : 34.20%


---
### **Conclusión Final**

In [3]:
# Recopilamos los datos exactos que han mostrado las celdas ejecutadas anteriores
datos_interpretabilidad = [
    {
        "Dataset": "Global Unificado",
        "Estrategia": "Fusión mediante Atención",
        "Vídeo": "29.53%",
        "Audio": "18.15%",
        "Texto": "52.32%"
    },
    
    {
        "Dataset": "MELD",
        "Estrategia": "Fusión Tardía (Regresión Logística)",
        "Vídeo": "7.49%",
        "Audio": "34.90%",
        "Texto": "57.60%"
    },
    {
        "Dataset": "IEMOCAP",
        "Estrategia": "Fusión Temprana",
        "Vídeo": "33.05%",
        "Audio": "32.75%",
        "Texto": "34.20%"
    }
]

# Creamos el DataFrame
df_resumen_interp = pd.DataFrame(datos_interpretabilidad)

# Lo guardamos en CSV 
df_resumen_interp.to_csv("resumen_final_interpretabilidad.csv", index=False)

styled_df = df_resumen_interp.style.set_properties(**{'text-align': 'center', 'background-color': '#f8f9fa', 'border': '1px solid black'})\
                                   .set_table_styles([dict(selector='th', props=[('text-align', 'center'), ('background-color', '#4C72B0'), ('color', 'white'), ('font-weight', 'bold')])])\
                                   .set_caption("Resumen Final de Interpretabilidad por Dominio")

display(styled_df)

,Dataset,Estrategia,Vídeo,Audio,Texto
0,Global Unificado,Fusión mediante Atención,29.53%,18.15%,52.32%
1,MELD,Fusión Tardía (Regresión Logística),7.49%,34.90%,57.60%
2,IEMOCAP,Fusión Temprana,33.05%,32.75%,34.20%


* **Fusión mediante Atención (Modelo en Dataset Global Unificado)**: El Texto vemos que domina con un 52.32%, seguido del Vídeo y finalmente del Audio, con un aporte este último muy pequeño a la decisión final. Al haber fusionado los datasets de MELD e IEMOCAP, las características acústicas y visuales son muy inestables, y la red aprende a anclarse a la modalidad más invariante y estable de todas, que es el **texto**. La semántica de las palabras de estrés se mantiene constante independientemente del micrófono o la cámara con la que se grabí la escena, conviertiendo al texto en el eje central para que el modelo permita generalizar sobre ambos dominios. 

* **Fusión Tardía mediante Regresión Logística (MELD)**: En este caso, el texto presenta una mayor importancia a la hora de tomar la decisión, y sorprende que en este caso, a diferencia del modelo global, el vídeo es el que menos importancia presenta. Como ya se indicó, el formato de los clips de vídeo de MELD presenta cortes abruptos, cambios de cámara, lo cual puede hacer que el modelo se confunda. 

* **Fusión Temprana (IEMOCAP)**: Los resultados son simétricos. Esta distribución equitativa (prácticamente de un 33% exacto para cada modalidad) es la confirmación de la gran estabilidad que presenta IEMOCAP. Cuando el actor simula estrés, altera su rostro, quiebra su voz y utiliza vocabulario tenso de forma simultánea, existe una **sincronía multimodal perfecta**. La red neuronal comprueba que las tres señales son de alta calidad, por lo que no necesita penalizar ni ignorar ninguna de ellas. Esto justifica la necesidad de la arquitectura de Fusión Temprana para este corpus. 